# Làm sạch dữ liệu GEO

In [ ]:
from pyspark.sql import SparkSession, functions as F

# Bước khởi tạo: tạo SparkSession để đọc dữ liệu GEO từ HDFS.
spark = (
    SparkSession.builder
    .appName("geo_cleaning")
    .getOrCreate()
)
sc = spark.sparkContext

# Bước khai báo đường dẫn: dữ liệu raw nằm trong HDFS tại /drugtarget/data/raw/geo.
# Với Spark, dùng URI HDFS đầy đủ để tránh bị hiểu nhầm thành thư mục local.
# Lưu ý: chỉ gán path vào biến để đọc, không tạo/sửa/ghi đè file raw.
HDFS_BASE_URI = "hdfs://master11:9000"
GEO_ROOT_RELATIVE = "/drugtarget/data/raw/geo"
GEO_ROOT = f"{HDFS_BASE_URI}{GEO_ROOT_RELATIVE}"
EXPRESSION_PATH = f"{GEO_ROOT}/expression/geo_ex.txt"
METADATA_PATH = f"{GEO_ROOT}/metadata"

# Bước cấu hình: đảm bảo Spark dùng đúng NameNode HDFS.
spark.sparkContext._jsc.hadoopConfiguration().set("fs.defaultFS", HDFS_BASE_URI)


def hdfs_exists(path: str) -> bool:
    """Kiểm tra một path HDFS có tồn tại hay không, không tạo/sửa file."""
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    return fs.exists(spark._jvm.org.apache.hadoop.fs.Path(path))


# Bước kiểm tra nguồn dữ liệu: báo lỗi sớm nếu thiếu expression hoặc metadata.
required_paths = [EXPRESSION_PATH, METADATA_PATH]
missing_paths = [path for path in required_paths if not hdfs_exists(path)]

if missing_paths:
    raise FileNotFoundError(
        "Không tìm thấy path HDFS bắt buộc: " + ", ".join(missing_paths)
    )

print(f"HDFS_BASE_URI     : {HDFS_BASE_URI}")
print(f"GEO_ROOT_RELATIVE : {GEO_ROOT_RELATIVE}")
print(f"GEO_ROOT          : {GEO_ROOT}")
print(f"EXPRESSION_PATH   : {EXPRESSION_PATH}")
print(f"METADATA_PATH     : {METADATA_PATH}")

## Bước 1 - In 10 dòng đầu

In [ ]:
# Bước 1.1: đọc expression raw dưới dạng từng dòng text, không sửa file gốc.
expression_raw = spark.read.text(EXPRESSION_PATH)

# Bước 1.2: in 10 dòng đầu để quan sát header và các dòng mô tả dư nếu có.
print("10 dòng đầu của expression raw:")
expression_raw.show(10, truncate=False)

In [ ]:
# Bước 1.3: đọc metadata raw từ thư mục metadata, vẫn chỉ đọc vào biến.
metadata_raw = spark.read.text(METADATA_PATH)

# Bước 1.4: in 10 dòng đầu để xem cấu trúc SOFT metadata của GEO.
print("10 dòng đầu của metadata raw:")
metadata_raw.show(10, truncate=False)

## Bước 2 - Xử lý dòng thừa nếu có

In [ ]:
# Bước 2.1: đánh số từng dòng expression để tìm vị trí header thật.
# File expression có thể có vài dòng mô tả ở đầu file. Header dữ liệu thật bắt đầu bằng:
# ID_REF<TAB>Description<TAB>sample_1...
expression_indexed = expression_raw.rdd.zipWithIndex().map(
    lambda row_idx: (row_idx[1], row_idx[0]["value"].rstrip("\r"))
)

# Bước 2.2: tìm header thật của bảng expression.
expression_header_candidates = (
    expression_indexed
    .filter(lambda item: item[1].startswith("ID_REF\tDescription\t"))
    .take(1)
)

if not expression_header_candidates:
    raise ValueError("Không tìm thấy header hợp lệ cho expression file")

expression_header_index = expression_header_candidates[0][0]

# Bước 2.3: gom các dòng dư trước header để kiểm tra, không xóa trong raw.
expression_extra_lines = (
    expression_indexed
    .filter(lambda item: item[0] < expression_header_index)
    .map(lambda item: item[1])
    .collect()
)

print(f"Số dòng thừa trước header expression: {len(expression_extra_lines)}")
for line in expression_extra_lines:
    print(repr(line))

# Bước 2.4: tạo RDD sạch trong biến riêng, bỏ dòng mô tả/blank trước header.
expression_clean_lines = (
    expression_indexed
    .filter(lambda item: item[0] >= expression_header_index)
    .map(lambda item: item[1])
    .filter(lambda line: line.strip() != "")
)

# Bước 2.5: parse RDD sạch thành DataFrame expression_df.
expression_df = (
    spark.read
    .option("sep", "\t")
    .option("header", True)
    .option("inferSchema", True)
    .option("nullValue", "NA")
    .option("nanValue", "NA")
    .csv(expression_clean_lines)
)

# Bước 2.6: xem lại 10 dòng đầu sau khi xử lý dòng thừa.
print(f"Expression rows: {expression_df.count():,}")
print(f"Expression columns: {len(expression_df.columns):,}")
expression_df.show(10, truncate=False)

In [ ]:
# Bước 2.7: đánh số từng dòng metadata để trích đúng bảng clinical annotations.
# Metadata GEO dạng SOFT có nhiều dòng mô tả; bảng cần dùng nằm giữa:
# !series_table_begin ... và !series_table_end
metadata_indexed = metadata_raw.rdd.zipWithIndex().map(
    lambda row_idx: (row_idx[1], row_idx[0]["value"].rstrip("\r"))
)

# Bước 2.8: tìm marker bắt đầu và kết thúc của bảng metadata.
metadata_begin = (
    metadata_indexed
    .filter(lambda item: item[1].startswith("!series_table_begin"))
    .take(1)
)
metadata_end = (
    metadata_indexed
    .filter(lambda item: item[1].startswith("!series_table_end"))
    .take(1)
)

if not metadata_begin or not metadata_end:
    raise ValueError("Không tìm thấy marker !series_table_begin/!series_table_end trong metadata")

metadata_table_start = metadata_begin[0][0] + 1
metadata_table_end = metadata_end[0][0]

# Bước 2.9: đếm các dòng mô tả bị bỏ qua trước/sau bảng, chỉ bỏ trong biến sạch.
metadata_extra_before = metadata_table_start
metadata_extra_after = metadata_indexed.filter(lambda item: item[0] > metadata_table_end).count()

print(f"Số dòng metadata bỏ qua trước bảng: {metadata_extra_before}")
print(f"Số dòng metadata bỏ qua sau bảng: {metadata_extra_after}")

# Bước 2.10: tạo RDD metadata sạch chỉ chứa bảng tabular.
metadata_clean_lines = (
    metadata_indexed
    .filter(lambda item: metadata_table_start <= item[0] < metadata_table_end)
    .map(lambda item: item[1])
    .filter(lambda line: line.strip() != "")
)

# Bước 2.11: parse metadata sạch thành DataFrame metadata_df.
metadata_df = (
    spark.read
    .option("sep", "\t")
    .option("header", True)
    .option("inferSchema", True)
    .option("nullValue", "NA")
    .option("nanValue", "NA")
    .csv(metadata_clean_lines)
)

# Bước 2.12: xem lại 10 dòng đầu metadata sau khi xử lý dòng thừa.
print(f"Metadata rows: {metadata_df.count():,}")
print(f"Metadata columns: {len(metadata_df.columns):,}")
metadata_df.show(10, truncate=False)

In [ ]:
# Bước kiểm tra cuối: so khớp sample trong expression với cột Array của metadata.
# Cell này chỉ tạo danh sách kiểm tra trong bộ nhớ, không ghi dữ liệu ra raw.
expression_sample_columns = [
    col_name
    for col_name in expression_df.columns
    if col_name not in {"ID_REF", "Description"}
]
metadata_sample_ids = {
    row["Array"]
    for row in metadata_df.select("Array").where(F.col("Array").isNotNull()).distinct().collect()
}

expression_only_samples = sorted(set(expression_sample_columns) - metadata_sample_ids)
metadata_only_samples = sorted(metadata_sample_ids - set(expression_sample_columns))

print(f"Sample columns trong expression: {len(expression_sample_columns):,}")
print(f"Sample IDs trong metadata: {len(metadata_sample_ids):,}")
print(f"Có trong expression nhưng thiếu metadata: {len(expression_only_samples):,}")
print(f"Có trong metadata nhưng thiếu expression: {len(metadata_only_samples):,}")

print("Ví dụ sample thiếu metadata:", expression_only_samples[:10])
print("Ví dụ metadata thiếu expression:", metadata_only_samples[:10])

## Bước 3 - Đổi hàng thành cột cho bảng expression

Mục tiêu: chuyển bảng expression từ dạng gene là hàng, bệnh nhân là cột sang dạng bệnh nhân là hàng, gene là cột.

In [ ]:
# Bước 3.1: kiểm tra ID_REF có bị trùng không trước khi đổi chiều bảng.
# ID_REF sẽ được dùng làm tên cột gene trong bảng expression mới.
duplicate_gene_ids = (
    expression_df
    .groupBy("ID_REF")
    .count()
    .where(F.col("count") > 1)
)

duplicate_gene_count = duplicate_gene_ids.count()
if duplicate_gene_count > 0:
    duplicate_gene_ids.show(20, truncate=False)
    raise ValueError(f"Có {duplicate_gene_count} ID_REF bị trùng, cần xử lý trước khi transpose")

# Bước 3.2: lấy danh sách cột bệnh nhân/sample trong bảng expression gốc.
# Hai cột ID_REF và Description là thông tin gene, không phải bệnh nhân.
patient_columns = [
    col_name
    for col_name in expression_df.columns
    if col_name not in {"ID_REF", "Description"}
]

# Bước 3.3: đưa các cột bệnh nhân về dạng long: Array, ID_REF, expression_value.
# Hàm stack chỉ tạo DataFrame mới trong bộ nhớ, không sửa file raw.
stack_expr = ", ".join([f"`{col_name}`, '{col_name}'" for col_name in patient_columns])
expression_long = expression_df.selectExpr(
    "ID_REF",
    f"stack({len(patient_columns)}, {stack_expr}) as (expression_value, Array)"
)

# Bước 3.4: tăng ngưỡng pivot vì số gene lớn hơn mặc định của Spark.
spark.conf.set("spark.sql.pivotMaxValues", "20000")

# Bước 3.5: pivot lại để mỗi bệnh nhân là một hàng và mỗi gene là một cột.
gene_columns = [row["ID_REF"] for row in expression_df.select("ID_REF").collect()]
expression = (
    expression_long
    .groupBy("Array")
    .pivot("ID_REF", gene_columns)
    .agg(F.first("expression_value"))
)

# Bước 3.6: kiểm tra nhanh kích thước bảng expression sau khi đổi chiều.
print(f"Số bệnh nhân trong expression: {expression.count():,}")
print(f"Số cột trong expression sau transpose: {len(expression.columns):,}")
expression.select(expression.columns[:10]).show(10, truncate=False)

## Bước 4 - Giữ thuộc tính metadata cần thiết và join với expression

Mục tiêu: tạo bảng `metadata_clean` chỉ gồm các thuộc tính cần giữ, sau đó join với bảng `expression` để tạo bảng `annotate`.

In [ ]:
# Bước 4.1: chuẩn hóa tên cột compos_agect/compos_agects nếu nguồn thay đổi tên.
# Dữ liệu hiện tại dùng cột compos_agects.
metadata_for_join = metadata_df
if "compos_agects" not in metadata_for_join.columns and "compos_agect" in metadata_for_join.columns:
    metadata_for_join = metadata_for_join.withColumnRenamed("compos_agect", "compos_agects")

# Bước 4.2: khai báo danh sách cột metadata cần giữ lại.
# Array được giữ thêm làm khóa bệnh nhân để join và nhận diện từng dòng.
metadata_keep_columns = [
    "Array",
    "Age",
    "Gender",
    "GenderCode",
    "Female",
    "Original Study",
    "MergeGroup",
    "Cohort",
    "Histology_MD",
    "Histology_progno",
    "Differentiation_MD",
    "DifferentiationNumeric",
    "GradeModerate",
    "GradePoor",
    "Stage_consensus_MD",
    "StageNumeric",
    "Stage_I_MD",
    "Stage_I_ADENO",
    "stageIB",
    "stageIIA",
    "stageIIB",
    "stageIIIA",
    "stageIIIB",
    "stageIV",
    "OS_Status",
    "OS_Time",
    "DSS_Status",
    "DSS_Time",
    "OSDSS_Status",
    "OSDSS_Time",
    "OSDSS60_Status",
    "OSDSS60_Time",
    "DEATH_before_5yrs",
    "DEATH_after_5yrs",
    "EXCLUDEFLAG",
    "trtest",
    "AgeGT70",
    "SEER_agects",
    "SEER_ageGT70",
    "scmodc20_sim",
    "compos_agects",
]

# Bước 4.3: kiểm tra cột còn thiếu để tránh join sai hoặc mất thuộc tính âm thầm.
missing_metadata_columns = [
    col_name
    for col_name in metadata_keep_columns
    if col_name not in metadata_for_join.columns
]

if missing_metadata_columns:
    raise ValueError("Thiếu các cột metadata bắt buộc: " + ", ".join(missing_metadata_columns))

# Bước 4.4: tạo bảng metadata sạch, chỉ giữ các cột cần thiết.
metadata_clean = metadata_for_join.select([F.col(col_name) for col_name in metadata_keep_columns])

print(f"Metadata clean rows: {metadata_clean.count():,}")
print(f"Metadata clean columns: {len(metadata_clean.columns):,}")
metadata_clean.show(10, truncate=False)

In [ ]:
# Bước 4.5: join metadata_clean với expression theo khóa bệnh nhân Array.
annotate = metadata_clean.join(expression, on="Array", how="inner")

# Bước 4.6: kiểm tra số dòng và khóa Array ở 3 bảng sạch.
def check_array_key(df, table_name: str) -> None:
    total_rows = df.count()
    null_array_rows = df.where(F.col("Array").isNull()).count()
    duplicated_array_rows = (
        df.groupBy("Array")
        .count()
        .where(F.col("count") > 1)
        .count()
    )
    print(f"{table_name}: rows={total_rows:,}, null Array={null_array_rows:,}, duplicated Array={duplicated_array_rows:,}")
    if null_array_rows > 0 or duplicated_array_rows > 0:
        raise ValueError(f"Bảng {table_name} có Array null hoặc bị trùng")


check_array_key(metadata_clean, "metadata_clean")
check_array_key(expression, "expression")
check_array_key(annotate, "annotate")

expected_patient_count = 1106
actual_counts = {
    "metadata_clean": metadata_clean.count(),
    "expression": expression.count(),
    "annotate": annotate.count(),
}

for table_name, row_count in actual_counts.items():
    if row_count != expected_patient_count:
        raise ValueError(
            f"Bảng {table_name} có {row_count:,} dòng, khác kỳ vọng {expected_patient_count:,}"
        )

print(f"Annotate rows: {annotate.count():,}")
print(f"Annotate columns: {len(annotate.columns):,}")
annotate.select(annotate.columns[:10]).show(10, truncate=False)

## Lưu 3 bảng sạch vào refined

Lưu `metadata_clean`, `expression`, và `annotate` dưới dạng CSV vào HDFS refined. Phần này chỉ ghi vào vùng refined, không ghi vào vùng raw.

In [ ]:
# Bước lưu 1: khai báo thư mục refined cho dữ liệu GEO.
REFINED_GEO_ROOT = f"{HDFS_BASE_URI}/drugtarget/data/refined/geo"

# Bước lưu 2: tạo thư mục root refined nếu chưa tồn tại.
# Lệnh mkdirs chỉ tác động vào vùng refined, không chạm vào raw.
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
fs.mkdirs(spark._jvm.org.apache.hadoop.fs.Path(REFINED_GEO_ROOT))

# Bước lưu 3: ghi từng bảng sạch thành CSV có header.
# mode("overwrite") giúp notebook chạy lại được và ghi đè đúng bảng refined cũ.
refined_tables = {
    "metadata": metadata_clean,
    "expression": expression,
    "annotate": annotate,
}

for table_name, df in refined_tables.items():
    output_path = f"{REFINED_GEO_ROOT}/{table_name}"
    (
        df.write
        .mode("overwrite")
        .option("header", True)
        .csv(output_path)
    )
    print(f"Đã lưu {table_name} vào {output_path}")

# Bước lưu 4: kiểm tra các thư mục output đã được tạo trong HDFS.
for table_name in refined_tables:
    output_path = f"{REFINED_GEO_ROOT}/{table_name}"
    if not hdfs_exists(output_path):
        raise FileNotFoundError(f"Chưa thấy thư mục output sau khi lưu: {output_path}")

print("Hoàn tất lưu 3 bảng refined: metadata, expression, annotate")